# Day 18 — Results Consolidation & Statistical Tests
**CST3990 Undergraduate Individual Project**  
Student: MANJOO Ameera Najla | M01014463  
Middlesex University Mauritius  

---
This notebook runs all Day 18 tasks:
1. Guard — verify Day 17 artefacts exist
2. Load per-clip FAR data
3. Display complete results tables (Blocks A–D)
4. **Test 1** — Wilcoxon signed-rank: `rule_based` vs `ocsvm` FAR (n=10)
5. **Test 2** — McNemar (exact): FAR-gate outcomes, `rule_based` vs `isolation_forest` (n=10)
6. **Test 3** — Pearson r: `pct_clipped` vs `far_value` across all 30 pairs
7. Visualisations: FAR box plots, correlation scatter
8. Failure cases documentation
9. Threats to validity
10. Write `day18_report.json` + `day18_statistical_results.csv`
11. Git commit

## Cell 1 — Mount Drive & set paths (Google Colab)

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, os

REPO_ROOT = '/content/drive/MyDrive/CST3990/CST-3990---Undergrad-Project'
SRC_DIR   = os.path.join(REPO_ROOT, 'src')
LOGS_DIR  = os.path.join(REPO_ROOT, 'logs')
BD_DIR    = os.path.join(LOGS_DIR, 'block_d')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print('Repo root:', REPO_ROOT)
print('src/ on path:', SRC_DIR)

## Cell 2 — Install dependencies

In [ ]:
# scipy and statsmodels are usually pre-installed in Colab; run if needed
import importlib, subprocess

for pkg, import_name in [('scipy', 'scipy'), ('statsmodels', 'statsmodels'),
                          ('matplotlib', 'matplotlib'), ('seaborn', 'seaborn')]:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(['pip', 'install', '-q', pkg])
        print(f'Installed {pkg}')
    else:
        print(f'{pkg}: already available')

## Cell 3 — Imports

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import wilcoxon, pearsonr
from statsmodels.stats.contingency_tables import mcnemar
from pathlib import Path

# Import project module
import day18_statistical_tests as d18

# Override LOGS_DIR so module resolves paths correctly inside Colab
import day18_statistical_tests as _m
_m.LOGS_DIR    = Path(LOGS_DIR)
_m.BLOCK_D_DIR = Path(BD_DIR)
_m.OUT_DIR     = Path(BD_DIR)

pd.set_option('display.max_colwidth', 80)
print('Imports OK')

## Cell 4 — Guard: verify Day 17 artefacts

In [ ]:
required_files = [
    os.path.join(BD_DIR, 'day17_far_per_clip.csv'),
    os.path.join(BD_DIR, 'block_d_complete_results.csv'),
    os.path.join(BD_DIR, 'day17_far_summary.csv'),
    os.path.join(BD_DIR, 'day17_latency_profile.json'),
    os.path.join(BD_DIR, 'block_d_event_predictions.csv'),
]

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = '✅' if exists else '❌ MISSING'
    print(f'{status}  {os.path.basename(f)}')
    if not exists:
        all_ok = False

if not all_ok:
    raise RuntimeError('One or more Day 17 artefacts are missing. Run Day 17 first.')

print('\n[Guard] All Day 17 artefacts present ✓')

## Cell 5 — Load per-clip FAR data

In [ ]:
df_far = d18._load_far_per_clip()

print(f'Shape: {df_far.shape}   (expected 30 rows: 10 clips × 3 methods)')
print(f'Clips  : {sorted(df_far.clip_filename.unique())}')
print(f'Methods: {sorted(df_far.method.unique())}')
print()

display(df_far.sort_values(['clip_filename', 'method']).reset_index(drop=True))

## Cell 6 — Complete results tables (Blocks A – D)

In [ ]:
tables = d18.build_complete_results_tables()

# ── Block A ──────────────────────────────────────────────────────────────────
print('=== Block A — Object Detection ===')
if 'error' not in tables['block_a']:
    df_a = pd.DataFrame(tables['block_a']['detectors'])
    display(df_a)
    print(f"Frozen: {tables['block_a']['frozen_detector']}")
else:
    print(tables['block_a']['error'])

# ── Block B ──────────────────────────────────────────────────────────────────
print('\n=== Block B — Multi-Object Tracking ===')
if 'error' not in tables['block_b']:
    df_b = pd.DataFrame(tables['block_b']['trackers'])
    display(df_b)
    print(f"Frozen: {tables['block_b']['frozen_tracker']}")
else:
    print(tables['block_b']['error'])

# ── Block C ──────────────────────────────────────────────────────────────────
print('\n=== Block C — Feature Extraction ===')
if 'error' not in tables['block_c']:
    df_c = pd.DataFrame(tables['block_c']['feature_sets'])
    display(df_c)
    print(f"Frozen: {tables['block_c']['frozen_feature_set']}")
else:
    print(tables['block_c']['error'])

# ── Block D ──────────────────────────────────────────────────────────────────
print('\n=== Block D — Anomaly Detection ===')
if 'error' not in tables['block_d']:
    df_d = pd.DataFrame(tables['block_d']['methods'])
    display(df_d[['method','total_events','events_detected','detection_coverage_pct',
                  'mean_far','std_far','max_far','far_gate_passed','notes']])
else:
    print(tables['block_d']['error'])

# ── Latency ──────────────────────────────────────────────────────────────────
print('\n=== Latency Profile ===')
if 'error' not in tables['latency']:
    lat = tables['latency']
    print(f"End-to-end FPS  : {lat['end_to_end_fps']}")
    print(f"Total mean ms   : {lat['total_mean_ms']}")
    print(f"Total p50 ms    : {lat['total_p50_ms']}")
    print(f"Total p95 ms    : {lat['total_p95_ms']}")
    print(f"Bottleneck stage: {lat['bottleneck_stage']} ({lat['anomaly_mean_ms']} ms mean)")
else:
    print(tables['latency']['error'])

## Cell 7 — Test 1: Wilcoxon signed-rank

**Hypothesis:** H₀: median(FAR_rule_based − FAR_ocsvm) = 0  
**Data:** Per-clip FAR values, 10 paired normal clips  
**Adaptation note:** Original plan used per-sequence mAP (yolo vs mobilenet).
Block A has only 2 aggregate rows. Per-clip FAR is the richest paired dataset.

In [ ]:
w_result = d18.run_wilcoxon_far_test(df_far)

print('── Wilcoxon signed-rank: rule_based vs OC-SVM FAR (n=10) ──')
print(f"  n pairs          : {w_result['n_pairs']}")
print(f"  Median FAR RB    : {w_result['median_rb']:.4f}")
print(f"  Median FAR OCSVM : {w_result['median_ocsvm']:.4f}")
print(f"  W statistic      : {w_result['statistic']:.4f}")
print(f"  p-value          : {w_result['p_value']:.4f}")
print(f"  Significant (α=0.05): {w_result['significant_05']}")
print()
print(f"  ► {w_result['interpretation']}")
print()
print(f"  Note: {w_result['note']}")

## Cell 8 — Test 2: McNemar (exact)

**Hypothesis:** H₀: P(IF-fail, RB-pass) = P(IF-pass, RB-fail)  
**Data:** Binary FAR-gate outcomes per clip (fail = FAR ≥ 0.10), 10 normal clips  
**Adaptation note:** Original plan used event-level predictions — all events detected by all methods, giving zero discordant pairs.

In [ ]:
m_result = d18.run_mcnemar_far_gate_test(df_far)

tbl = m_result['contingency_table']
ct = np.array([[tbl['both_pass_a'],   tbl['if_fail_rb_pass_b']],
               [tbl['if_pass_rb_fail_c'], tbl['both_fail_d']]])

print('── McNemar (exact): rule_based vs Isolation Forest FAR-gate (n=10) ──')
print(f"  Threshold = {m_result['threshold']}")
print()
print('  Contingency table  (rows = IF outcome, cols = RB outcome, 0=pass, 1=fail):')
print(f'  [[both pass, IF-fail RB-pass],  = [[{ct[0,0]}, {ct[0,1]}],')
print(f'   [IF-pass RB-fail, both fail]]     [{ct[1,0]}, {ct[1,1]}]]')
print()
print(f"  Discordant b (IF-fail, RB-pass)  : {m_result['discordant_b']}")
print(f"  Discordant c (IF-pass, RB-fail)  : {m_result['discordant_c']}")
print(f"  p-value (exact)                  : {m_result['p_value']:.4f}")
print(f"  Significant (α=0.05)             : {m_result['significant_05']}")
print()
print(f"  ► {m_result['interpretation']}")
print()
print(f"  Note: {m_result['note']}")

## Cell 9 — Test 3: Pearson r

**Hypothesis:** H₀: ρ(pct_clipped, far_value) = 0  
**Data:** All 30 method-clip pairs (10 clips × 3 methods)  
**Adaptation note:** Original plan used per-sequence IDSW vs speed_std (n=2 — statistically useless).

In [ ]:
p_result = d18.run_pearson_domain_shift_test(df_far)

print('── Pearson r: pct_clipped vs far_value (n=30) ──')
print(f"  n pairs        : {p_result['n_pairs']}")
print(f"  r              : {p_result['r']:.4f}")
print(f"  r²             : {p_result['r_squared']:.4f}")
print(f"  p-value        : {p_result['p_value']:.4f}")
print(f"  Significant (α=0.05): {p_result['significant_05']}")
print()
print(f"  ► {p_result['interpretation']}")
print()
print(f"  Note: {p_result['note']}")

## Cell 10 — Statistical summary table

In [ ]:
summary = pd.DataFrame([
    {
        'Test':          'Wilcoxon signed-rank',
        'Comparison':    'rule_based vs OC-SVM FAR (n=10)',
        'Statistic':     f"W = {w_result['statistic']:.2f}",
        'p-value':       f"{w_result['p_value']:.4f}",
        'Sig. (α=0.05)': '✓' if w_result['significant_05'] else '✗',
        'Effect':        f"Δmedian = {w_result['median_rb']-w_result['median_ocsvm']:.4f}",
    },
    {
        'Test':          'McNemar (exact)',
        'Comparison':    'rule_based vs IF FAR-gate (n=10)',
        'Statistic':     f"b={m_result['discordant_b']}, c={m_result['discordant_c']}",
        'p-value':       f"{m_result['p_value']:.4f}",
        'Sig. (α=0.05)': '✓' if m_result['significant_05'] else '✗',
        'Effect':        f"b/(b+c) = {m_result['discordant_b']/(m_result['discordant_b']+max(m_result['discordant_c'],1)):.2f}",
    },
    {
        'Test':          'Pearson r',
        'Comparison':    'pct_clipped vs FAR (n=30)',
        'Statistic':     f"r = {p_result['r']:.4f}",
        'p-value':       f"{p_result['p_value']:.4f}",
        'Sig. (α=0.05)': '✓' if p_result['significant_05'] else '✗',
        'Effect':        f"r² = {p_result['r_squared']:.4f}",
    },
])

display(summary)

## Cell 11 — Visualisation: FAR distributions by method

In [ ]:
METHOD_COLOURS = {
    'rule_based':       '#2196F3',   # blue
    'ocsvm':            '#4CAF50',   # green
    'isolation_forest': '#FF5722',   # red-orange
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Block D — FAR across Normal Clips\n'
             'CST3990 | MANJOO Ameera Najla | M01014463', fontsize=13, fontweight='bold')

# ── Panel 1: box plot ────────────────────────────────────────────────────────
ax = axes[0]
methods_order = ['rule_based', 'ocsvm', 'isolation_forest']
data_bp = [df_far[df_far.method == m]['far_value'].values for m in methods_order]
bp = ax.boxplot(data_bp, patch_artist=True, widths=0.5, notch=False)
for patch, meth in zip(bp['boxes'], methods_order):
    patch.set_facecolor(METHOD_COLOURS[meth])
    patch.set_alpha(0.75)
ax.axhline(0.10, color='crimson', linestyle='--', linewidth=1.5, label='FAR threshold (0.10)')
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Rule-Based', 'OC-SVM', 'Isolation\nForest'], fontsize=9)
ax.set_ylabel('FAR value')
ax.set_title('Box plot — FAR per clip')
ax.legend(fontsize=8)
ax.set_ylim(-0.02, 1.05)

# ── Panel 2: strip / swarm ───────────────────────────────────────────────────
ax = axes[1]
for meth in methods_order:
    sub = df_far[df_far.method == meth]
    ax.scatter(
        [meth] * len(sub),
        sub['far_value'],
        color=METHOD_COLOURS[meth],
        alpha=0.85,
        s=60,
        zorder=3,
    )
ax.axhline(0.10, color='crimson', linestyle='--', linewidth=1.5, label='Threshold 0.10')
ax.set_ylabel('FAR value')
ax.set_title('Strip plot — per-clip FAR values')
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=8)
ax.set_xticklabels(['Rule-Based', 'OC-SVM', 'Isolation\nForest'], fontsize=9)

# ── Panel 3: domain shift scatter ────────────────────────────────────────────
ax = axes[2]
for meth in methods_order:
    sub = df_far[df_far.method == meth]
    ax.scatter(
        sub['pct_clipped'],
        sub['far_value'],
        color=METHOD_COLOURS[meth],
        label=meth,
        alpha=0.80,
        s=55,
    )
# Regression line
from numpy.polynomial.polynomial import polyfit
x_all = df_far['pct_clipped'].values
y_all = df_far['far_value'].values
b0, b1 = np.polyfit(x_all, y_all, 1)
x_line = np.linspace(x_all.min(), x_all.max(), 100)
ax.plot(x_line, b0 * x_line + b1, 'k--', linewidth=1.5,
        label=f'OLS fit (r={p_result["r"]:.3f}, p={p_result["p_value"]:.3f})')
ax.axhline(0.10, color='crimson', linestyle=':', linewidth=1.2)
ax.set_xlabel('pct_clipped (domain shift proxy)')
ax.set_ylabel('FAR value')
ax.set_title('Pearson r: domain shift vs FAR')
ax.legend(fontsize=7)

plt.tight_layout()

fig_path = os.path.join(BD_DIR, 'day18_far_visualisations.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

## Cell 12 — Wilcoxon per-pair detail

In [ ]:
# Pair-level breakdown for Wilcoxon
pivot = df_far.pivot_table(
    index='clip_filename', columns='method', values='far_value', aggfunc='first'
).reset_index()

pivot['diff_rb_minus_ocsvm'] = pivot['rule_based'] - pivot['ocsvm']
pivot['rb_gate'] = pivot['rule_based'].apply(lambda x: 'PASS' if x < 0.10 else 'FAIL')
pivot['if_gate'] = pivot['isolation_forest'].apply(lambda x: 'PASS' if x < 0.10 else 'FAIL')
pivot['ocsvm_gate'] = pivot['ocsvm'].apply(lambda x: 'PASS' if x < 0.10 else 'FAIL')

display(pivot[[
    'clip_filename', 'rule_based', 'ocsvm', 'isolation_forest',
    'diff_rb_minus_ocsvm', 'rb_gate', 'ocsvm_gate', 'if_gate'
]].round(4))

## Cell 13 — McNemar per-clip detail

In [ ]:
mc_detail = pd.DataFrame(m_result['per_clip_detail'])
mc_detail['rb_fail_label']  = mc_detail['rb_fail'].map({True: 'FAIL', False: 'PASS'})
mc_detail['if_fail_label']  = mc_detail['if_fail'].map({True: 'FAIL', False: 'PASS'})
mc_detail['discordant']     = mc_detail['rb_fail'] != mc_detail['if_fail']

display(mc_detail[[
    'clip', 'rb_far', 'if_far', 'rb_fail_label', 'if_fail_label', 'discordant'
]].rename(columns={
    'rb_far':       'FAR (rule_based)',
    'if_far':       'FAR (IF)',
    'rb_fail_label':'RB gate',
    'if_fail_label':'IF gate',
}).round(4))

n_discord = mc_detail['discordant'].sum()
print(f'\nTotal discordant pairs: {n_discord} / {len(mc_detail)}')

## Cell 14 — Failure cases documentation

In [ ]:
failures = d18.document_failure_cases()

df_fail = pd.DataFrame([
    {
        'ID':          f['id'],
        'Block':       f['block'],
        'Description': f['description'][:90] + '...' if len(f['description']) > 90 else f['description'],
        'Impact':      f['impact'][:80] + '...'      if len(f['impact']) > 80      else f['impact'],
        'Resolution':  f['resolution'][:80] + '...'  if len(f['resolution']) > 80  else f['resolution'],
    }
    for f in failures
])

display(df_fail)
print(f'\nTotal documented failure/adaptation cases: {len(failures)}')

## Cell 15 — Threats to validity

In [ ]:
# Build a fake report just to extract threats
_rpt = d18.generate_day18_report(w_result, m_result, p_result, tables, failures)
threats = _rpt['threats_to_validity']

for i, t in enumerate(threats, 1):
    print(f"[T{i}] {t['threat']}")
    print(f"      Description : {t['description']}")
    print(f"      Mitigation  : {t['mitigation']}")
    print()

## Cell 16 — Write day18_report.json & day18_statistical_results.csv

In [ ]:
report = d18.generate_day18_report(w_result, m_result, p_result, tables, failures)

report_path = os.path.join(BD_DIR, 'day18_report.json')
with open(report_path, 'w', encoding='utf-8') as fh:
    json.dump(report, fh, indent=2, default=str)
print(f'Report written → {report_path}')

stats_rows = [
    {
        'test':           'Wilcoxon signed-rank',
        'comparison':     'rule_based vs ocsvm FAR per clip',
        'n':              w_result['n_pairs'],
        'statistic':      w_result['statistic'],
        'p_value':        w_result['p_value'],
        'significant_05': w_result['significant_05'],
        'effect_size':    f"Δmedian={w_result['median_rb']-w_result['median_ocsvm']:.4f}",
        'data_source':    'day17_far_per_clip.csv',
    },
    {
        'test':           'McNemar (exact)',
        'comparison':     'rule_based vs IF FAR-gate outcomes',
        'n':              m_result['n_clips'],
        'statistic':      m_result['statistic'],
        'p_value':        m_result['p_value'],
        'significant_05': m_result['significant_05'],
        'effect_size':    f"b={m_result['discordant_b']}, c={m_result['discordant_c']}",
        'data_source':    'day17_far_per_clip.csv',
    },
    {
        'test':           'Pearson r',
        'comparison':     'pct_clipped vs far_value (all 30 pairs)',
        'n':              p_result['n_pairs'],
        'statistic':      p_result['r'],
        'p_value':        p_result['p_value'],
        'significant_05': p_result['significant_05'],
        'effect_size':    f"r²={p_result['r_squared']:.4f}",
        'data_source':    'day17_far_per_clip.csv',
    },
]

df_stats = pd.DataFrame(stats_rows)
stats_path = os.path.join(BD_DIR, 'day18_statistical_results.csv')
df_stats.to_csv(stats_path, index=False)
print(f'Stats CSV written → {stats_path}')

display(df_stats)

## Cell 17 — Git commit

In [ ]:
import subprocess

def _git(cmd: list[str]) -> str:
    result = subprocess.run(
        ['git', '-C', REPO_ROOT] + cmd,
        capture_output=True, text=True
    )
    return (result.stdout + result.stderr).strip()

files_to_add = [
    'logs/block_d/day18_report.json',
    'logs/block_d/day18_statistical_results.csv',
    'logs/block_d/day18_far_visualisations.png',
    'src/day18_statistical_tests.py',
    'notebooks/day18_results_consolidation.ipynb',
]

for f in files_to_add:
    out = _git(['add', f])
    if out:
        print(out)

commit_msg = (
    'Day 18: Results consolidation + statistical tests\n\n'
    '- Wilcoxon: rule_based vs OC-SVM per-clip FAR (n=10)\n'
    '- McNemar (exact): rule_based vs IF FAR-gate outcomes (n=10)\n'
    '- Pearson r: domain shift (pct_clipped) vs FAR (n=30)\n'
    '- Complete results tables (Blocks A-D) + latency profile\n'
    '- Failure cases documented (F1, F2, F3, F14, F29, STAT-1/2/3)\n'
    '- Threats to validity updated\n'
    '- day18_report.json + day18_statistical_results.csv written'
)

print(_git(['status', '--short']))
out = _git(['commit', '-m', commit_msg])
print(out)

print('\n[Day 18] Git commit done ✓')

## Cell 18 — Day 18 complete ✓

```
Day 18 — Results Consolidation & Statistical Tests   ✓ COMPLETE
─────────────────────────────────────────────────────────────────
  Test 1 — Wilcoxon signed-rank  (rule_based vs OC-SVM FAR, n=10)
  Test 2 — McNemar exact         (rule_based vs IF FAR-gate, n=10)
  Test 3 — Pearson r             (pct_clipped vs FAR, n=30)
  Outputs: day18_report.json, day18_statistical_results.csv
           day18_far_visualisations.png
─────────────────────────────────────────────────────────────────
PROJECT COMPLETE  ✓  All 18 days executed.
```

**Frozen pipeline:**
- **Block A** Detector: `yolov8n`  mAP@0.5 = 0.6674
- **Block B** Tracker: `sort`      IDF1 = 0.0412
- **Block C** Features: `F2_only`  AUROC = 0.9935
- **Block D** Anomaly: `OC-SVM` + `rule_based` (IF: FAR gate failed, annotated F14)
- **Latency**: 21.11 FPS end-to-end (bottleneck: anomaly stage 43.48 ms mean)